In [4]:
import praw
import pandas as pd
import os

from dotenv import load_dotenv

load_dotenv()

False

In [6]:
from datasets import load_dataset

dataset = load_dataset("sm4rtdev/reddit_dataset_255")

c:\Users\USER\Downloads\Trending Project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\USER\Downloads\Trending Project\venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\USER\.cache\huggingface\hub\datasets--sm4rtdev--reddit_dataset_255. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator

In [7]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'dataType', 'communityName', 'datetime', 'username_encoded', 'url_encoded'],
        num_rows: 124331
    })
})


In [8]:
print(dataset["train"].column_names)

['text', 'label', 'dataType', 'communityName', 'datetime', 'username_encoded', 'url_encoded']


In [9]:
reddit_sample = dataset["train"].select(range(5000))

In [10]:
reddit_df = reddit_sample.to_pandas()

In [11]:
print(reddit_df.shape)

(5000, 7)


In [12]:
reddit_df.head()

,text,label,dataType,communityName,datetime,username_encoded,url_encoded
0,http://spaceinimages.esa.int/Images/2012/12/Pr...,r/spaceengineering,comment,r/SpaceEngineering,2013-01-14,Z0FBQUFBQm9WUzJLS04wZGN4MHZqYkJBOUJKVUplNThzTk...,Z0FBQUFBQm9WUzJLZHNQOXlaaEREWWVFMk9fRmZwSldGSH...
1,from http://www.reddit.com/r/MachinePorn/comme...,r/spaceengineering,comment,r/SpaceEngineering,2013-01-21,Z0FBQUFBQm9WUzJLaFBRdm1GdDN6VjJTYzNYdlhYT09qUH...,Z0FBQUFBQm9WUzJLOHRwMEZwdFdOUkNqYUt1Q0pXejctN0...
2,I had the idea for a reading list related to v...,r/spaceexploration,post,r/Spaceexploration,2014-06-21,Z0FBQUFBQm9WUzJLbFZBZDQzN1NMMkptckh2WUxISklBZD...,Z0FBQUFBQm9WUzJLRlN5RnNwOHhHOC1vd3JBM0ZZOWlTZz...
3,Looks like the proton rocket issue has been re...,r/spaceengineering,comment,r/SpaceEngineering,2016-02-04,Z0FBQUFBQm9WUzJLNFM5TTlVRXowR1BFM0dCWkdyTWlOVF...,Z0FBQUFBQm9WUzJLb1JZbHhRQjJyYklVYk5pNDZwWkNab0...
4,I believe the focus in space engineering going...,r/spaceengineering,post,r/SpaceEngineering,2016-02-25,Z0FBQUFBQm9WUzJLX0tJdVlhWHg0ajlZMmNsYkJ1VkhUb1...,Z0FBQUFBQm9WUzJLbGYzd0paRlRtemk1cEt3OU9Hb3l6dH...


In [13]:
reddit_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   text              5000 non-null   str  
 1   label             5000 non-null   str  
 2   dataType          5000 non-null   str  
 3   communityName     5000 non-null   str  
 4   datetime          5000 non-null   str  
 5   username_encoded  5000 non-null   str  
 6   url_encoded       5000 non-null   str  
dtypes: str(7)
memory usage: 3.8 MB


In [14]:
print(reddit_df["dataType"].value_counts())

dataType
comment    4472
post        528
Name: count, dtype: int64


In [15]:
reddit_df["datetime"] = pd.to_datetime(
    reddit_df["datetime"],
    errors="coerce"
)

In [16]:
print(reddit_df["datetime"].min())
print(reddit_df["datetime"].max())

2013-01-14 00:00:00
2025-06-20 00:00:00


In [17]:
print(reddit_df["datetime"].isna().sum())

0


In [18]:
import re

def clean_text(text):
    text = str(text)
    
    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)
    
    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text)
    
    return text.strip()

In [19]:
reddit_df["clean_text"] = reddit_df["text"].apply(clean_text)

In [20]:
reddit_df[["text", "clean_text"]].head(10)

,text,clean_text
0,http://spaceinimages.esa.int/Images/2012/12/Pr...,
1,from http://www.reddit.com/r/MachinePorn/comme...,from
2,I had the idea for a reading list related to v...,I had the idea for a reading list related to v...
3,Looks like the proton rocket issue has been re...,Looks like the proton rocket issue has been re...
4,I believe the focus in space engineering going...,I believe the focus in space engineering going...
5,Here is a related article by the co-author:\nh...,"Here is a related article by the co-author: , ..."
6,I came across this notion and I'm not sure to ...,I came across this notion and I'm not sure to ...
7,It's a technology or change to the status quo ...,It's a technology or change to the status quo ...
8,Thanks appreciated!,Thanks appreciated!
9,Many thanks to @inharmsway for allowing me to ...,Many thanks to @inharmsway for allowing me to ...


In [21]:
reddit_df = reddit_df[
    reddit_df["clean_text"].str.len() > 0
].copy()

In [22]:
print("Remaining records:", len(reddit_df))

Remaining records: 4986


In [23]:
reddit_df = reddit_df.drop_duplicates(
    subset=["clean_text"]
).copy()

In [24]:
print("After removing duplicates:", len(reddit_df))

After removing duplicates: 4867


In [25]:
reddit_df.to_csv(
    "../data/reddit_cleaned.csv",
    index=False
)

In [26]:
print(reddit_df["communityName"].value_counts().head(20))

communityName
r/AskReddit            397
r/NoStupidQuestions    337
r/wallstreetbets       330
r/ufc                  268
r/Monopoly_GO          261
r/nba                  241
r/space                201
r/soccer               199
r/politics             198
r/AITAH                196
r/spacex               177
r/BlueOrigin           170
r/Bitcoin              166
r/nasa                 147
r/SquaredCircle        137
r/AmIOverreacting      126
r/Spaceexploration     123
r/SpaceXStarship       120
r/CryptoCurrency       114
r/mac                  108
Name: count, dtype: int64


In [27]:
print(
    reddit_df.groupby(
        reddit_df["datetime"].dt.to_period("M")
    ).size().head(20)
)

datetime
2013-01    1
2014-06    1
2016-02    2
2016-11    1
2017-01    3
2017-03    1
2017-08    1
2017-12    1
2018-02    1
2018-06    1
2018-11    4
2018-12    1
2019-05    1
2019-06    2
2019-10    1
2019-11    2
2020-04    1
2020-06    3
2020-07    2
2020-08    1
Freq: M, dtype: int64


In [28]:
reddit_df["month"] = reddit_df["datetime"].dt.to_period("M")

In [29]:
monthly_volume = (
    reddit_df
    .groupby("month")
    .size()
    .reset_index(name="post_count")
)

In [30]:
monthly_volume.head()

,month,post_count
0,2013-01,1
1,2014-06,1
2,2016-02,2
3,2016-11,1
4,2017-01,3


In [31]:
monthly_volume.head()

,month,post_count
0,2013-01,1
1,2014-06,1
2,2016-02,2
3,2016-11,1
4,2017-01,3


In [2]:
from datasets import load_dataset

dataset = load_dataset("sm4rtdev/reddit_dataset_255")

In [3]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'dataType', 'communityName', 'datetime', 'username_encoded', 'url_encoded'],
        num_rows: 124331
    })
})


In [4]:
print(dataset["train"].column_names)

['text', 'label', 'dataType', 'communityName', 'datetime', 'username_encoded', 'url_encoded']


In [5]:
reddit_sample = dataset["train"].select(range(5000))
reddit_df = reddit_sample.to_pandas()

print(reddit_df.shape)

(5000, 7)


In [6]:
print(reddit_df.head())

                                                text               label  \
0  http://spaceinimages.esa.int/Images/2012/12/Pr...  r/spaceengineering   
1  from http://www.reddit.com/r/MachinePorn/comme...  r/spaceengineering   
2  I had the idea for a reading list related to v...  r/spaceexploration   
3  Looks like the proton rocket issue has been re...  r/spaceengineering   
4  I believe the focus in space engineering going...  r/spaceengineering   

  dataType       communityName    datetime  \
0  comment  r/SpaceEngineering  2013-01-14   
1  comment  r/SpaceEngineering  2013-01-21   
2     post  r/Spaceexploration  2014-06-21   
3  comment  r/SpaceEngineering  2016-02-04   
4     post  r/SpaceEngineering  2016-02-25   

                                    username_encoded  \
0  Z0FBQUFBQm9WUzJLS04wZGN4MHZqYkJBOUJKVUplNThzTk...   
1  Z0FBQUFBQm9WUzJLaFBRdm1GdDN6VjJTYzNYdlhYT09qUH...   
2  Z0FBQUFBQm9WUzJLbFZBZDQzN1NMMkptckh2WUxISklBZD...   
3  Z0FBQUFBQm9WUzJLNFM5TTlVRXowR1BFM0dCWkd

In [7]:
print(reddit_df["dataType"].value_counts())

dataType
comment    4472
post        528
Name: count, dtype: int64


In [8]:
print(reddit_df["communityName"].value_counts().head(20))

communityName
r/AskReddit            400
r/NoStupidQuestions    338
r/wallstreetbets       335
r/Monopoly_GO          323
r/ufc                  270
r/nba                  242
r/space                202
r/soccer               200
r/politics             200
r/AITAH                196
r/spacex               180
r/BlueOrigin           173
r/Bitcoin              166
r/nasa                 155
r/SquaredCircle        141
r/AmIOverreacting      126
r/Spaceexploration     124
r/SpaceXStarship       123
r/CryptoCurrency       114
r/mac                  111
Name: count, dtype: int64


In [12]:
import pandas as pd

reddit_df["datetime"] = pd.to_datetime(
    reddit_df["datetime"],
    errors="coerce"
)

print("Min datetime:", reddit_df["datetime"].min())
print("Max datetime:", reddit_df["datetime"].max())
print("Invalid datetime rows:", reddit_df["datetime"].isna().sum())

Min datetime: 2013-01-14 00:00:00
Max datetime: 2025-06-20 00:00:00
Invalid datetime rows: 0


In [13]:
from datasets import load_dataset

dataset = load_dataset("sm4rtdev/reddit_dataset_255")

In [14]:
reddit_full = dataset["train"].to_pandas()

print(reddit_full.shape)

(124331, 7)


In [15]:
import pandas as pd

reddit_full["datetime"] = pd.to_datetime(
    reddit_full["datetime"],
    errors="coerce"
)

print(reddit_full["datetime"].min())
print(reddit_full["datetime"].max())

2013-01-14 00:00:00
2025-06-21 00:00:00


In [16]:
print(reddit_full["dataType"].value_counts())

dataType
comment    122608
post         1723
Name: count, dtype: int64


Your 5,000-row sample had:

comment    4472
post        528

So approximately 89% comments / 11% posts in your sample.

That's important.

For our project, we'll use:

posts + comments for topic/aggression analysis
but we'll keep dataType so we know what each record is.

In [17]:
dataset["train"].select(range(5000))

Dataset({
    features: ['text', 'label', 'dataType', 'communityName', 'datetime', 'username_encoded', 'url_encoded'],
    num_rows: 5000
})

In [18]:
reddit_df = reddit_full.sample(
    n=5000,
    random_state=42
).copy()

In [19]:
print(reddit_df["datetime"].min())
print(reddit_df["datetime"].max())

2020-09-17 00:00:00
2025-06-21 00:00:00


In [20]:
print(
    reddit_df["communityName"]
    .value_counts()
    .head(20)
)

communityName
r/politics             1279
r/worldnews             973
r/wallstreetbets        538
r/technology            322
r/wow                   226
r/Bitcoin               172
r/CryptoCurrency        135
r/comics                 93
r/AITAH                  76
r/vegan                  75
r/mac                    73
r/comicbooks             73
r/Monopoly_GO            71
r/apple                  69
r/nba                    67
r/NoStupidQuestions      67
r/teenagers              63
r/AskReddit              53
r/gadgets                51
r/keto                   44
Name: count, dtype: int64


In [21]:
import re

def clean_text(text):
    text = str(text)

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()

Cleaning of REddit data set

In [22]:
reddit_df["clean_text"] = reddit_df["text"].apply(clean_text)

In [23]:
reddit_df = reddit_df[
    reddit_df["clean_text"].str.len() > 0
].copy()

In [24]:
print("Remaining rows:", len(reddit_df))

Remaining rows: 4987


In [25]:
reddit_df = reddit_df.drop_duplicates(
    subset=["clean_text"]
).copy()

print("After duplicates:", len(reddit_df))

After duplicates: 4934


In [26]:
reddit_df["month"] = reddit_df["datetime"].dt.to_period("M")

In [27]:
print(reddit_df["month"].head())

81068     2025-06
15510     2025-06
109943    2025-06
59409     2025-06
4878      2025-06
Name: month, dtype: period[M]


In [28]:
reddit_df.to_csv(
    "../data/reddit_cleaned.csv",
    index=False
)